# Notebook 02 — LLM Schema Mapping Agent (Layer 2, Agent 1)

**Abstract:** Main experiment: mapping accuracy of LLaMA 3 8B vs Gemini/Claude API
across 4 datasets of increasing complexity.
Implements **Confidence-Gated Routing (Innovation #1)** and
**Adaptive Few-Shot Memory via RAG+FAISS (Innovation #2)**.

**References:**
- CHESS: Talaei et al. (2024) — Contextual Harnessing for Efficient SQL Synthesis
- Semantic-RAG: Birjega (2025) — Semantic Schema Mapping
- Annam (2025) — LLM-Powered Autonomous Agents for ETL Pipeline Generation

In [ ]:
# Install required packages into the active kernel environment (run once)
%pip install -q lxml matplotlib seaborn pandas numpy scipy pydantic requests tqdm httpx faiss-cpu sentence-transformers google-generativeai


In [ ]:
# Cell 1 — Setup
%matplotlib inline
import sys, os, json, time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

def _find_research_root() -> str:
    sentinel = "generate_datasets.py"
    candidate = os.path.abspath(os.getcwd())
    for _ in range(5):
        if os.path.exists(os.path.join(candidate, sentinel)):
            return candidate
        parent = os.path.dirname(candidate)
        if parent == candidate:
            break
        candidate = parent
    sub = os.path.join(os.path.abspath(os.getcwd()), "research")
    if os.path.exists(os.path.join(sub, sentinel)):
        return sub
    raise FileNotFoundError(
        f"Could not locate research root (sentinel '{sentinel}' not found). "
        "Run from bi-platform/ or bi-platform/research/."
    )

RESEARCH_ROOT = _find_research_root()
os.chdir(RESEARCH_ROOT)
if RESEARCH_ROOT not in sys.path:
    sys.path.insert(0, RESEARCH_ROOT)
print(f"RESEARCH_ROOT = {RESEARCH_ROOT}")

from src.ingestion import MultiSourceIngester
from src.profiler import SchemaProfiler
from src.llm_client import LLMClient, MockLLMClient
from src.schema_mapper import SchemaMapper
from src.rag import RAGSchemaStore
from src.evaluator import ETLEvaluator
from src.visualizer import ResearchVisualizer
from data.ground_truth.ground_truth import GROUND_TRUTH

ingester = MultiSourceIngester()
profiler = SchemaProfiler()
evaluator = ETLEvaluator()
viz = ResearchVisualizer()
rag_store = RAGSchemaStore()

datasets = ingester.ingest_all()

try:
    real_client = LLMClient()
    llm_client = real_client if real_client.is_ollama_available() else MockLLMClient()
except Exception:
    llm_client = MockLLMClient()
print(f"LLM client: {type(llm_client).__name__}")

mapper = SchemaMapper(llm_client=llm_client, rag_store=rag_store)
print(f"RAG store size: {rag_store.size} examples")


In [ ]:
# Cell 2 — Prompt template display
# Show the exact prompt for dataset1
first_key = list(datasets.keys())[0]
first_df = datasets[first_key]
first_ctx = profiler.profile(first_df, first_key.split('.')[0])

prompt_no_fewshot = mapper.build_prompt(first_ctx, k_shots=0)
prompt_with_fewshot = mapper.build_prompt(first_ctx, k_shots=3)

print('=== PROMPT WITHOUT FEW-SHOT ===')
print(prompt_no_fewshot[:500])
print('...')
print()
print('=== PROMPT WITH 3 FEW-SHOT EXAMPLES ===')
print(prompt_with_fewshot[:800])
print('...')

In [ ]:
# Cell 3 — Run mapping on all 4 datasets under 3 conditions
gt_keys = {
    'dataset1_retail_sales': 'dataset1_retail_sales',
    'dataset2_hospital_records': 'dataset2_hospital_records',
    'dataset3_supplier_invoices': 'dataset3_supplier_invoices',
    'dataset4_ecommerce_events': 'dataset4_ecommerce_events',
}

conditions = ['llama_only', 'llama_fewshot', 'routed']
all_results = {cond: {} for cond in conditions}
all_confidence = []
all_accuracy = []

for fname, df in datasets.items():
    short = fname.split('.')[0]
    gt_key = gt_keys.get(short)
    if not gt_key or gt_key not in GROUND_TRUTH:
        continue
    
    gt = GROUND_TRUTH[gt_key]
    ctx = profiler.profile(df, short)
    
    for cond in conditions:
        # Reset random seed for reproducibility per condition
        import random
        random.seed(42 + hash(cond + short) % 1000)
        
        result = mapper.map_schema(
            ctx,
            condition=cond,
            k_shots=3 if cond != 'llama_only' else 0,
            difficulty=gt['difficulty'],
        )
        
        predicted = {
            'fact_table': result.fact_table,
            'dimensions': result.dimensions,
            'measures': result.measures,
        }
        acc = evaluator.mapping_accuracy(predicted, gt)
        
        all_results[cond][short] = {
            'accuracy': acc,
            'confidence': result.confidence,
            'model_used': result.model_used,
            'latency_ms': result.latency_ms,
            'fallback_reason': result.fallback_reason,
            'predicted': predicted,
        }
        
        if cond == 'routed':
            all_confidence.append(result.confidence)
            all_accuracy.append(acc)
        
        print(f"{short} [{cond}]: acc={acc:.3f}, conf={result.confidence:.2f}, "
              f"model={result.model_used}, lat={result.latency_ms:.0f}ms")

In [ ]:
# Cell 4 — Results table
rows = []
ds_names = list(all_results['llama_only'].keys())
for ds in ds_names:
    row = {'Dataset': ds}
    for cond in conditions:
        row[f'{cond}_acc'] = all_results[cond][ds]['accuracy']
    rows.append(row)

results_df = pd.DataFrame(rows)
results_df.columns = ['Dataset', 'A: LLaMA Only', 'B: LLaMA+FewShot', 'C: Routed']

# Color gradient display
def color_acc(val):
    if isinstance(val, (int, float)):
        color = f'background-color: rgba({int(255*(1-val))}, {int(255*val)}, 0, 0.3)'
        return color
    return ''

styled = results_df.style.applymap(color_acc, subset=['A: LLaMA Only', 'B: LLaMA+FewShot', 'C: Routed'])
display(styled)

In [ ]:
# Cell 5 — Routing analysis for Condition C
print('=== Routing Analysis (Condition C: Routed) ===')
routing_data = {}
llama_latencies = []
claude_latencies = []

for ds in ds_names:
    r = all_results['routed'][ds]
    model = r['model_used']
    reason = r.get('fallback_reason', 'direct')
    print(f"  {ds}: model={model}, confidence={r['confidence']:.2f}, reason={reason}")
    
    routing_data[ds] = {
        'llama_only': 1 if model == 'llama3' else 0,
        'llama_fallback': 1 if 'fallback' in str(reason) or model == 'claude' else 0,
        'claude_only': 0,  # Not in our routing setup
    }
    
    if 'llama' in model.lower():
        llama_latencies.append(r['latency_ms'])
    else:
        claude_latencies.append(r['latency_ms'])

fig = viz.plot_routing_distribution(routing_data)
display(fig); plt.close(fig)
print('Saved: results/figures/fig2_routing_distribution.pdf')

In [ ]:
# Cell 6 — Confidence calibration plot
fig = viz.plot_confidence_vs_accuracy(all_confidence, all_accuracy)
display(fig); plt.close(fig)

if len(all_confidence) > 1:
    r_val = np.corrcoef(all_confidence, all_accuracy)[0, 1]
    print(f'Pearson correlation: r = {r_val:.3f}')
    if r_val > 0.5:
        print('→ Model confidence is well-calibrated (r > 0.5)')
    else:
        print('→ Model confidence needs improvement (r ≤ 0.5)')
print('Saved: results/figures/fig3_confidence_calibration.pdf')

In [ ]:
# Cell 7 — Qualitative analysis (dataset4 = hard)
hard_ds = 'dataset4_ecommerce_events'
if hard_ds in all_results['routed']:
    r = all_results['routed'][hard_ds]
    gt = GROUND_TRUTH[hard_ds]
    print(f'=== Qualitative Analysis: {hard_ds} (Hard) ===')
    print(f'\nLLM Output (raw):')
    print(json.dumps(r['predicted'], indent=2))
    print(f'\nGround Truth:')
    print(json.dumps({k: gt[k] for k in ['fact_table', 'dimensions', 'measures']}, indent=2))
    print(f'\nAccuracy: {r["accuracy"]:.3f}')
    print(f'Model used: {r["model_used"]}')
    print(f'Confidence: {r["confidence"]:.2f}')
    
    # Highlight correct/incorrect
    pred = r['predicted']
    print(f'\nFact table: {"✓" if pred["fact_table"] == gt["fact_table"] else "✗"} '
          f'(predicted: {pred["fact_table"]}, expected: {gt["fact_table"]})')
    
    pred_dims = set(pred['dimensions'])
    gt_dims = set(gt['dimensions'])
    print(f'Dimensions correct: {pred_dims & gt_dims}')
    print(f'Dimensions missing: {gt_dims - pred_dims}')
    print(f'Dimensions extra (hallucinated): {pred_dims - gt_dims}')

In [ ]:
# Cell 8 — Save metrics
os.makedirs('results/metrics', exist_ok=True)

metrics = {
    'per_dataset_accuracy': {},
    'routing_distribution': {},
    'avg_confidence_per_dataset': {},
    'avg_latency_llama_ms': float(np.mean(llama_latencies)) if llama_latencies else 0,
    'avg_latency_claude_ms': float(np.mean(claude_latencies)) if claude_latencies else 0,
    'all_confidence_scores': all_confidence,
    'all_accuracy_scores': all_accuracy,
}

for ds in ds_names:
    metrics['per_dataset_accuracy'][ds] = {
        cond: all_results[cond][ds]['accuracy'] for cond in conditions
    }
    metrics['avg_confidence_per_dataset'][ds] = all_results['routed'][ds]['confidence']
    metrics['routing_distribution'][ds] = routing_data.get(ds, {})

# Confidence calibration
if len(all_confidence) > 1:
    metrics['confidence_calibration_r'] = float(np.corrcoef(all_confidence, all_accuracy)[0, 1])
else:
    metrics['confidence_calibration_r'] = 0.0

with open('results/metrics/02_schema_mapping.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved: results/metrics/02_schema_mapping.json')
print('\n✓ Notebook 02 complete.')